# 02 — Learning vs exact: Levels 1-4

Re-entrant: loads sweep results from `results/main` (H200) when present,
else `results/smoke`; if neither exists, runs the smoke stage inline. Then
trains one demo policy for the Level-3 policy-space comparison and one
unregularised small-n policy for the Level-4 overlearning exhibit.
Scales switch automatically on GPU.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, jax, jax.numpy as jnp
import matplotlib.pyplot as plt
import mfoptmm as mf
from mfoptmm.hjb import make_ops, at_state, interp_axis
from mfoptmm.policy_eval import policy_value, myopic_policy, repeat_block_start
FIGS = ROOT / "docs" / "figures"; FIGS.mkdir(parents=True, exist_ok=True)
print("device:", jax.devices()[0].platform)
from mfoptmm import compare
GPU = jax.devices()[0].platform != "cpu" 

In [ ]:
RESULTS = ROOT/"results"/"main"
if not any(RESULTS.glob("*.npz")):
    RESULTS = ROOT/"results"/"smoke"
if not any(RESULTS.glob("*.npz")):
    print("no results found - running the smoke stage inline (minutes)")
    import subprocess
    subprocess.run([sys.executable, str(ROOT/"tests"/"run_h200.py"),
                    "--stage", "smoke", "--trials", "2"], cwd=ROOT, check=True)
    RESULTS = ROOT/"results"/"smoke"
results = compare.load_trials(RESULTS)
print(f"{len(results)} trials from {RESULTS}\n")
print(compare.gap_table(results))

In [ ]:
# Levels 1-2: gap figures
compare.plot_gap_vs_n(results, FIGS/"level1_gen_gap.png", level="gen")
compare.plot_gap_vs_n(results, FIGS/"level2_opt_gap.png", level="opt")
from IPython.display import Image, display
display(Image(str(FIGS/"level1_gen_gap.png")))
display(Image(str(FIGS/"level2_opt_gap.png")))

In [ ]:
# Level 3: demo policy vs the exact a1 solution
K_blocks, r, epochs = (36, 256, 2000) if GPU else (4, 32, 300)
spec = mf.bbg_example(variant="a1", K_blocks=K_blocks, profile="replication")
sol = mf.hjb_solve(spec, store_quotes=True)
pol = mf.train(spec, mf.TrainConfig(r=r, n=32, epochs=epochs, lr0=0.3, lr1=3e-3, seed=0))
compare.plot_policy_surfaces(pol, spec, sol.quotes, FIGS/"level3_policy.png")
display(Image(str(FIGS/"level3_policy.png")))
errs = compare.policy_errors(pol, spec, sol.quotes)
Kk = np.asarray(spec.book.strike); Tm = np.asarray(spec.book.mat)
print("quote errors in spread units 1/b (sup / visitation-weighted L2):")
for kv, tv in ((12., 1.), (10., 1.), (8., 1.)):
    i = int(np.where((Kk == kv) & (Tm == tv))[0][0])
    print(f"  ({kv:.0f},{tv:g}): {errs['sup'][i]:.2f} / {errs['l2_visit'][i]:.2f}")

In [ ]:
# Level 4: unregularised, small n - inventory paths, train vs fresh environments
pol_u = mf.train(spec, mf.TrainConfig(r=r, n=8, epochs=epochs, lr0=0.3, lr1=3e-3,
                                      seed=8, regularised=False))
compare.plot_overlearning_paths(pol_u, spec, seed=8, out=FIGS/"level4_overlearning.png")
display(Image(str(FIGS/"level4_overlearning.png")))

## Scaling up

The headline science runs on the H200:

```bash
python tests/run_h200.py --stage main --trials 20    # K=36, r=256, 20k epochs
python tests/run_h200.py --stage wide --trials 8     # r=1024 subset
```

Both are resumable (per-trial `.npz` checkpoints). Rerunning this notebook
afterwards picks up `results/main` automatically. The main stage adds: the
gap-vs-n^{-1} law at 20 trials/point, the `beta_n14` temperature arm
(beta = 100 (n/32)^{1/4}), and the unregularised arm at scale.